# 🛒 05 — Market Basket Analysis

> **Objective**: Discover products frequently purchased together using Association Rule Mining — the same technique used by Amazon, Walmart, and Flipkart for product recommendations.

**Algorithm**: Apriori (from `mlxtend`)

**Key Metrics:**
- **Support**: How often items appear together
- **Confidence**: P(B | A) — if A is bought, probability of B
- **Lift**: How much more likely A and B are bought together vs independently (Lift > 1 = positive association)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
COLORS = {'primary': '#003B73', 'secondary': '#0074B7', 'success': '#27AE60', 'alert': '#BF212F'}
print("Libraries loaded")

## 1. Load & Prepare Transaction Data

In [ ]:
df = pd.read_csv('../../02_Dataset/cleaned_data/sales_clean.csv')
print(f"Total transactions: {df.shape[0]:,}")
print(f"Unique orders: {df['order_id'].nunique():,}")
print(f"Unique sub-categories: {df['sub_category'].nunique()}")

# Use sub_category for basket analysis (product_name has too many unique values)
# Group items by order_id
basket = df.groupby('order_id')['sub_category'].apply(list).reset_index()
basket.columns = ['order_id', 'items']

# Filter orders with 2+ items
basket['item_count'] = basket['items'].apply(len)
multi_item = basket[basket['item_count'] >= 2].copy()
print(f"\nOrders with 2+ items: {len(multi_item):,} ({len(multi_item)/len(basket)*100:.1f}%)")

## 2. One-Hot Encoding (Transaction Matrix)

In [ ]:
# Remove duplicates within each order
transactions = multi_item['items'].apply(lambda x: list(set(x))).tolist()

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

print(f"Transaction matrix: {basket_df.shape}")
print(f"\nItem frequency (% of baskets):")
item_freq = basket_df.mean().sort_values(ascending=False) * 100
print(item_freq.round(2).to_string())

In [ ]:
# Item Frequency Bar Chart
fig, ax = plt.subplots(figsize=(14, 6))
item_freq.sort_values().plot.barh(ax=ax, color=COLORS['secondary'], edgecolor='white')
ax.set_title('Sub-Category Frequency in Multi-Item Orders', fontsize=14, fontweight='bold')
ax.set_xlabel('Frequency (%)')
for i, v in enumerate(item_freq.sort_values()):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../visualizations/item_frequency.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Apriori — Frequent Itemsets

In [ ]:
frequent = apriori(basket_df, min_support=0.02, use_colnames=True, max_len=3)
frequent['length'] = frequent['itemsets'].apply(len)
frequent = frequent.sort_values('support', ascending=False)

print(f"Frequent itemsets found: {len(frequent)}")
print(f"  2-item sets: {(frequent['length'] == 2).sum()}")
print(f"  3-item sets: {(frequent['length'] == 3).sum()}")
print(f"\nTop 15 Frequent Itemsets:")
top15 = frequent.head(15).copy()
top15['itemsets_str'] = top15['itemsets'].apply(lambda x: ' + '.join(sorted(x)))
print(top15[['itemsets_str', 'support', 'length']].to_string(index=False))

## 4. Association Rules

In [ ]:
rules = association_rules(frequent, metric='lift', min_threshold=1.0, num_itemsets=len(frequent))
rules = rules.sort_values('lift', ascending=False)

# Format for display
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))

print(f"Association rules found: {len(rules)}")
print(f"\nTop 20 Rules by Lift:")
display_cols = ['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']
print(rules[display_cols].head(20).to_string(index=False))

In [ ]:
# Scatter plot: Support vs Confidence, colored by Lift
fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(rules['support'], rules['confidence'], c=rules['lift'],
                     cmap='YlOrRd', s=rules['lift'] * 30, alpha=0.7, edgecolors='white')
plt.colorbar(scatter, label='Lift', ax=ax)
ax.set_xlabel('Support', fontsize=12)
ax.set_ylabel('Confidence', fontsize=12)
ax.set_title('Association Rules — Support vs Confidence (Size & Color = Lift)', fontsize=14, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Confidence = 50%')
ax.legend()
plt.tight_layout()
plt.savefig('../visualizations/basket_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Top Product Bundles

In [ ]:
# High-confidence, high-lift rules
strong_rules = rules[(rules['confidence'] >= 0.3) & (rules['lift'] >= 1.2)].copy()
strong_rules = strong_rules.sort_values('lift', ascending=False)

print(f"Strong rules (confidence >= 30%, lift >= 1.2): {len(strong_rules)}")
print("\nRecommended Product Bundles:")
print("=" * 80)
for _, row in strong_rules.head(10).iterrows():
    print(f"  If customer buys: {row['antecedents_str']}")
    print(f"  Recommend:        {row['consequents_str']}")
    print(f"  Confidence: {row['confidence']:.1%} | Lift: {row['lift']:.2f} | Support: {row['support']:.3f}")
    print("-" * 80)

In [ ]:
# Network visualization of top rules
fig, ax = plt.subplots(figsize=(14, 8))
top_rules = strong_rules.head(15)
y_pos = range(len(top_rules))
labels = [f"{r['antecedents_str']} -> {r['consequents_str']}" for _, r in top_rules.iterrows()]
colors = ['#27AE60' if l >= 1.5 else '#F39C12' if l >= 1.2 else '#95A5A6' for l in top_rules['lift']]
ax.barh(y_pos, top_rules['lift'], color=colors, edgecolor='white', height=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Lift', fontsize=12)
ax.set_title('Top Association Rules by Lift', fontsize=14, fontweight='bold')
ax.axvline(1, color='red', linestyle='--', alpha=0.7, label='Lift = 1 (no association)')
ax.legend()
plt.tight_layout()
plt.savefig('../visualizations/top_association_rules.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Export Results

In [ ]:
export = rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'leverage', 'conviction']].copy()
export.columns = ['Antecedent', 'Consequent', 'Support', 'Confidence', 'Lift', 'Leverage', 'Conviction']
export = export.sort_values('Lift', ascending=False)
export.to_csv('../outputs/association_rules.csv', index=False)
print(f"Exported {len(export):,} association rules to outputs/association_rules.csv")

## 7. Key Insights & Business Recommendations

| # | Finding | Action |
|---|---|---|
| 1 | Certain sub-categories are **frequently bought together** | Create product bundles |
| 2 | High-lift rules indicate **genuine associations** (not just popularity) | Use for recommendation engines |
| 3 | Rules with **confidence > 50%** are strong candidates for cross-selling | Feature in "Customers also bought" sections |
| 4 | Low-support high-lift rules may represent **niche opportunities** | Target via personalized email campaigns |
| 5 | The analysis mirrors techniques used by **Amazon and Walmart** | Demonstrates industry-standard analytics skills |

---
**Advanced Analytics Phase Complete!**

### Outputs Generated:
- `outputs/rfm_scores.csv` — RFM scores for all customers
- `outputs/customer_segments.csv` — K-Means cluster labels + RFM segments
- `outputs/association_rules.csv` — Product association rules

### Next Step: `06_Machine_Learning/` — Sales Forecasting with Prophet & XGBoost